In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# Step11.74 Python MCP-counter
# Reload standardized expression matrix
# Run in Python (TCGA)
# ============================================================

import os
import pandas as pd
import numpy as np

project_dir = _hcc_path('')

step11_dir = os.path.join(
    project_dir,
    "LIHC_RNA_Step11_Immune"
)

input_dir = os.path.join(
    step11_dir,
    "Input"
)

result_dir = os.path.join(
    step11_dir,
    "Results"
)

fig_dir = os.path.join(
    step11_dir,
    "Figures"
)

rna_mcp = pd.read_csv(
    os.path.join(
        input_dir,
        "LIHC_Expression_365_log2CPM.tsv"
    ),
    sep="\t",
    index_col=0
)

print("RNA:", rna_mcp.shape)
print("Duplicated genes:", rna_mcp.index.duplicated().sum())
print("Missing:", rna_mcp.isna().sum().sum())

In [ ]:
# ============================================================
# Step11.75 Download official MCP-counter marker table
# ============================================================

import urllib.request

marker_url = (
    "https://raw.githubusercontent.com/"
    "ebecht/MCPcounter/master/Signatures/genes.txt"
)

marker_file = os.path.join(
    input_dir,
    "MCPcounter_official_genes.txt"
)

urllib.request.urlretrieve(
    marker_url,
    marker_file
)

print("Saved:")
print(marker_file)

In [ ]:
# ============================================================
# Step11.76 Read official MCP-counter markers
# ============================================================

mcp_markers = pd.read_csv(
    marker_file,
    sep="\t",
    quotechar='"'
)

print(mcp_markers.shape)
print(mcp_markers.columns.tolist())

display(mcp_markers.head(10))

In [ ]:
# ============================================================
# Step11.77 MCP-counter marker coverage
# ============================================================

coverage_list = []

for cell_type, sub in mcp_markers.groupby("Cell population"):

    genes = (
        sub["HUGO symbols"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    present = [
        g for g in genes
        if g in rna_mcp.index
    ]

    coverage_list.append({
        "Cell_population": cell_type,
        "Official_markers": len(genes),
        "Present_markers": len(present),
        "Missing_markers": len(genes) - len(present),
        "Coverage": len(present) / len(genes)
    })

mcp_coverage = pd.DataFrame(
    coverage_list
).sort_values("Cell_population")

display(mcp_coverage)

In [ ]:
mcp_coverage.to_csv(
    os.path.join(
        result_dir,
        "LIHC_MCPcounter_Marker_Coverage.csv"
    ),
    index=False
)

In [ ]:
# ============================================================
# Step11.78 Calculate MCP-counter scores
# Official logic:
# mean expression of available marker genes per population
# ============================================================

mcp_scores_dict = {}

for cell_type, sub in mcp_markers.groupby("Cell population"):

    genes = (
        sub["HUGO symbols"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    present = [
        g for g in genes
        if g in rna_mcp.index
    ]

    if len(present) == 0:
        print("No markers found:", cell_type)
        continue

    # Official appendSignatures logic:
    # mean expression across marker genes for each sample
    score = (
        rna_mcp
        .loc[present]
        .mean(axis=0)
    )

    mcp_scores_dict[cell_type] = score


mcp_score = pd.DataFrame(
    mcp_scores_dict
)

mcp_score.index.name = "Sample_ID"

print("MCP-counter score shape:")
print(mcp_score.shape)

print("\nPopulations:")
print(mcp_score.columns.tolist())

display(mcp_score.head())

In [ ]:
# ============================================================
# Step11.79 Save Python MCP-counter results
# ============================================================

mcp_score.to_csv(
    os.path.join(
        result_dir,
        "LIHC_MCPcounter_Python_Scores_365.csv"
    )
)

print("MCP-counter scores saved.")

In [ ]:
# ============================================================
# Step11.80 Merge MCP-counter scores with RiskScore/RiskGroup
# ============================================================

import os
import pandas as pd
import numpy as np

risk_immune = pd.read_csv(
    os.path.join(
        input_dir,
        "LIHC_RiskGroup_365.csv"
    ),
    index_col=0
)

risk_immune.index.name = "Sample_ID"
mcp_score.index.name = "Sample_ID"

print("MCP:", mcp_score.shape)
print("Risk:", risk_immune.shape)

common_samples = mcp_score.index.intersection(
    risk_immune.index
)

print("Common samples:", len(common_samples))

mcp_merge = (
    mcp_score.loc[common_samples]
    .join(
        risk_immune.loc[
            common_samples,
            ["RiskScore", "RiskGroup", "OS_time", "OS"]
        ]
    )
)

print("Merged:", mcp_merge.shape)

print(
    mcp_merge["RiskGroup"].value_counts()
)

display(mcp_merge.head())

In [ ]:
# ============================================================
# Step11.81 MCP-counter High vs Low
# ============================================================

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


mcp_cell_cols = mcp_score.columns.tolist()

group_results = []


for cell in mcp_cell_cols:

    high = mcp_merge.loc[
        mcp_merge["RiskGroup"] == "High",
        cell
    ].dropna()

    low = mcp_merge.loc[
        mcp_merge["RiskGroup"] == "Low",
        cell
    ].dropna()

    stat, p = mannwhitneyu(
        high,
        low,
        alternative="two-sided"
    )

    group_results.append({
        "Cell_population": cell,
        "High_mean": high.mean(),
        "Low_mean": low.mean(),
        "High_median": high.median(),
        "Low_median": low.median(),
        "Median_difference_High_minus_Low":
            high.median() - low.median(),
        "U_statistic": stat,
        "P_value": p
    })


mcp_group_stats = pd.DataFrame(
    group_results
)


mcp_group_stats["FDR"] = multipletests(
    mcp_group_stats["P_value"],
    method="fdr_bh"
)[1]


mcp_group_stats = (
    mcp_group_stats
    .sort_values(["FDR", "P_value"])
    .reset_index(drop=True)
)


display(mcp_group_stats)

In [ ]:
mcp_group_stats.to_csv(
    os.path.join(
        result_dir,
        "LIHC_MCPcounter_High_vs_Low.csv"
    ),
    index=False
)

In [ ]:
# ============================================================
# Step11.82 MCP-counter vs RiskScore
# ============================================================

from scipy.stats import spearmanr


corr_results = []


for cell in mcp_cell_cols:

    temp = mcp_merge[
        [cell, "RiskScore"]
    ].dropna()

    rho, p = spearmanr(
        temp[cell],
        temp["RiskScore"]
    )

    corr_results.append({
        "Cell_population": cell,
        "Spearman_rho": rho,
        "P_value": p
    })


mcp_corr_stats = pd.DataFrame(
    corr_results
)


mcp_corr_stats["FDR"] = multipletests(
    mcp_corr_stats["P_value"],
    method="fdr_bh"
)[1]


mcp_corr_stats = (
    mcp_corr_stats
    .sort_values(["FDR", "P_value"])
    .reset_index(drop=True)
)


display(mcp_corr_stats)

In [ ]:
mcp_corr_stats.to_csv(
    os.path.join(
        result_dir,
        "LIHC_MCPcounter_RiskScore_Correlation.csv"
    ),
    index=False
)

In [ ]:
# ============================================================
# Step11.83 Cross-method direction concordance
# Robust version
# ============================================================

mapping = {
    "B_cells": "B lineage",
    "CD8_T_cells": "CD8 T cells",
    "Cytotoxic_cells": "Cytotoxic lymphocytes",
    "NK_cells": "NK cells",
    "Neutrophils": "Neutrophils"
}


comparison_rows = []


for ssgsea_cell, mcp_cell in mapping.items():

    s1 = ssgsea_group.loc[
        ssgsea_group["Cell_type"] == ssgsea_cell
    ].iloc[0]

    s2 = ssgsea_corr.loc[
        ssgsea_corr["Cell_type"] == ssgsea_cell
    ].iloc[0]

    m1 = mcp_group_stats.loc[
        mcp_group_stats["Cell_population"] == mcp_cell
    ].iloc[0]

    m2 = mcp_corr_stats.loc[
        mcp_corr_stats["Cell_population"] == mcp_cell
    ].iloc[0]


    # 不依赖之前保存的差值列名，直接现算
    ss_diff = (
        s1["High_median"]
        -
        s1["Low_median"]
    )

    mcp_diff = (
        m1["High_median"]
        -
        m1["Low_median"]
    )


    comparison_rows.append({

        "Immune_feature": ssgsea_cell,

        "ssGSEA_High_median": s1["High_median"],
        "ssGSEA_Low_median": s1["Low_median"],
        "ssGSEA_High_minus_Low": ss_diff,
        "ssGSEA_Group_FDR": s1["FDR"],

        "ssGSEA_Risk_rho": s2["Spearman_rho"],
        "ssGSEA_Risk_FDR": s2["FDR"],


        "MCP_High_median": m1["High_median"],
        "MCP_Low_median": m1["Low_median"],
        "MCP_High_minus_Low": mcp_diff,
        "MCP_Group_FDR": m1["FDR"],

        "MCP_Risk_rho": m2["Spearman_rho"],
        "MCP_Risk_FDR": m2["FDR"],


        "Group_direction_consistent":
            np.sign(ss_diff) == np.sign(mcp_diff),

        "Risk_direction_consistent":
            np.sign(s2["Spearman_rho"])
            ==
            np.sign(m2["Spearman_rho"])
    })


cross_method = pd.DataFrame(
    comparison_rows
)

display(cross_method)

In [ ]:
cross_method.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ssGSEA_vs_MCPcounter_Concordance.csv"
    ),
    index=False
)

print("Saved.")

In [ ]:
# ============================================================
# Step11.84 MCP-counter High vs Low boxplots
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import os

mcp_cells = mcp_score.columns.tolist()

fig, axes = plt.subplots(
    2,
    5,
    figsize=(18, 8)
)

axes = axes.flatten()


for ax, cell in zip(axes, mcp_cells):

    high = mcp_merge.loc[
        mcp_merge["RiskGroup"] == "High",
        cell
    ].dropna()

    low = mcp_merge.loc[
        mcp_merge["RiskGroup"] == "Low",
        cell
    ].dropna()


    ax.boxplot(
        [low, high],
        labels=["Low", "High"],
        showfliers=False
    )


    stat_row = mcp_group_stats.loc[
        mcp_group_stats["Cell_population"] == cell
    ].iloc[0]

    fdr = stat_row["FDR"]


    if fdr < 0.001:
        ptxt = "FDR < 0.001"
    else:
        ptxt = f"FDR = {fdr:.3f}"


    ax.set_title(
        cell,
        fontsize=11
    )

    ax.text(
        0.5,
        0.95,
        ptxt,
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=9
    )

    ax.set_ylabel(
        "MCP-counter score"
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


fig.suptitle(
    "MCP-counter immune populations by risk group",
    fontsize=15
)

fig.tight_layout(
    rect=[0, 0, 1, 0.96]
)


fig.savefig(
    os.path.join(
        fig_dir,
        "MCPcounter_High_vs_Low.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "MCPcounter_High_vs_Low.pdf"
    ),
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.85 UPDATED V2
# MCP-counter vs RiskScore correlation
# Cleaner publication-style lollipop plot
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt


plot_df = (
    mcp_corr_stats
    .sort_values("Spearman_rho")
    .reset_index(drop=True)
)

y = np.arange(len(plot_df))


fig, ax = plt.subplots(
    figsize=(9.5, 6.5)
)


# ============================================================
# 1. Lollipop lines
# ============================================================

ax.hlines(
    y=y,
    xmin=0,
    xmax=plot_df["Spearman_rho"],
    linewidth=1.4
)


# ============================================================
# 2. Points
# ============================================================

ax.scatter(
    plot_df["Spearman_rho"],
    y,
    s=75,
    zorder=3
)


# ============================================================
# 3. Zero reference line
# ============================================================

ax.axvline(
    0,
    linestyle="--",
    linewidth=1
)


# ============================================================
# 4. Y axis
# ============================================================

ax.set_yticks(y)

ax.set_yticklabels(
    plot_df["Cell_population"],
    fontsize=10
)


# ============================================================
# 5. Leave a clean right-side region for FDR labels
# ============================================================

xmin = -0.205
xmax = 0.205

ax.set_xlim(
    xmin,
    xmax
)


# ============================================================
# 6. FDR labels
# Fixed position on the RIGHT
# ============================================================

fdr_x = 0.115

for i, row in plot_df.iterrows():

    fdr = row["FDR"]

    if fdr < 0.001:
        label = "FDR < 0.001"
    else:
        label = f"FDR = {fdr:.3f}"

    ax.text(
        fdr_x,
        i,
        label,
        va="center",
        ha="left",
        fontsize=9,
        fontweight="bold" if fdr < 0.05 else "normal"
    )


# ============================================================
# 7. Add rho values beside each point
# ============================================================

for i, row in plot_df.iterrows():

    rho = row["Spearman_rho"]

    # negative values: label slightly to LEFT of point
    if rho < 0:

        ax.text(
            rho - 0.006,
            i,
            f"{rho:.3f}",
            va="center",
            ha="right",
            fontsize=8
        )

    # positive values: label slightly to RIGHT
    else:

        ax.text(
            rho + 0.006,
            i,
            f"{rho:.3f}",
            va="center",
            ha="left",
            fontsize=8
        )


# ============================================================
# 8. Labels
# ============================================================

ax.set_xlabel(
    "Spearman correlation with RiskScore",
    fontsize=11
)

ax.set_title(
    "MCP-counter immune populations vs RiskScore",
    fontsize=14,
    pad=12
)


# ============================================================
# 9. Clean appearance
# ============================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(
    axis="both",
    labelsize=10
)


fig.tight_layout()


# ============================================================
# 10. SAVE BEFORE SHOW
# ============================================================

fig.savefig(
    os.path.join(
        fig_dir,
        "MCPcounter_RiskScore_Correlation_FINAL.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "MCPcounter_RiskScore_Correlation_FINAL.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.86
# ssGSEA vs MCP-counter cross-method concordance
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# 为了显示更漂亮
plot_df = cross_method.copy()

name_map = {
    "B_cells": "B cells",
    "CD8_T_cells": "CD8 T cells",
    "Cytotoxic_cells": "Cytotoxic cells",
    "NK_cells": "NK cells",
    "Neutrophils": "Neutrophils"
}

plot_df["Display_name"] = (
    plot_df["Immune_feature"]
    .map(name_map)
)


# 只画 RiskScore Spearman rho
y = np.arange(len(plot_df))


fig, ax = plt.subplots(
    figsize=(8, 5.5)
)


offset = 0.15


ax.scatter(
    plot_df["ssGSEA_Risk_rho"],
    y + offset,
    s=80,
    label="ssGSEA"
)

ax.scatter(
    plot_df["MCP_Risk_rho"],
    y - offset,
    s=80,
    marker="s",
    label="MCP-counter"
)


# 每个细胞两种算法之间连线
for i, row in plot_df.iterrows():

    ax.plot(
        [
            row["ssGSEA_Risk_rho"],
            row["MCP_Risk_rho"]
        ],
        [
            i + offset,
            i - offset
        ],
        linewidth=1
    )


ax.axvline(
    0,
    linestyle="--",
    linewidth=1
)


ax.set_yticks(y)

ax.set_yticklabels(
    plot_df["Display_name"]
)

ax.set_xlabel(
    "Spearman correlation with RiskScore"
)

ax.set_title(
    "Cross-method validation of immune-cell associations"
)

ax.legend(
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "ssGSEA_vs_MCPcounter_RiskScore_Concordance.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "ssGSEA_vs_MCPcounter_RiskScore_Concordance.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.87
# Cross-method summary table
# ============================================================

summary_rows = []


for _, row in cross_method.iterrows():

    ss_dir = (
        "Higher in High-risk"
        if row["ssGSEA_High_minus_Low"] > 0
        else "Lower in High-risk"
    )

    mcp_dir = (
        "Higher in High-risk"
        if row["MCP_High_minus_Low"] > 0
        else "Lower in High-risk"
    )


    if (
        row["ssGSEA_Group_FDR"] < 0.05
        and
        row["MCP_Group_FDR"] < 0.05
        and
        row["Group_direction_consistent"]
    ):
        group_validation = "Validated"

    elif row["Group_direction_consistent"]:
        group_validation = "Direction consistent"

    else:
        group_validation = "Discordant"


    if (
        row["ssGSEA_Risk_FDR"] < 0.05
        and
        row["MCP_Risk_FDR"] < 0.05
        and
        row["Risk_direction_consistent"]
    ):
        corr_validation = "Validated"

    elif row["Risk_direction_consistent"]:
        corr_validation = "Direction consistent"

    else:
        corr_validation = "Discordant"


    summary_rows.append({

        "Immune_feature":
            row["Immune_feature"],

        "ssGSEA_group":
            ss_dir,

        "MCP_group":
            mcp_dir,

        "Group_validation":
            group_validation,

        "ssGSEA_rho":
            row["ssGSEA_Risk_rho"],

        "MCP_rho":
            row["MCP_Risk_rho"],

        "Correlation_validation":
            corr_validation
    })


validation_summary = pd.DataFrame(
    summary_rows
)


display(validation_summary)

In [ ]:
validation_summary.to_csv(
    os.path.join(
        result_dir,
        "LIHC_Immune_CrossMethod_Validation_Summary.csv"
    ),
    index=False
)

print("Cross-method validation summary saved.")

In [ ]:
#ssGSEA vs MCP-counter 双算法一致性总结图

In [ ]:
# ============================================================
# Step11.86
# ssGSEA vs MCP-counter RiskScore correlation concordance
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


plot_df = cross_method.copy()

name_map = {
    "B_cells": "B cells",
    "CD8_T_cells": "CD8 T cells",
    "Cytotoxic_cells": "Cytotoxic cells",
    "NK_cells": "NK cells",
    "Neutrophils": "Neutrophils"
}

plot_df["Display_name"] = (
    plot_df["Immune_feature"]
    .map(name_map)
)

# 固定顺序
order = [
    "B cells",
    "CD8 T cells",
    "Cytotoxic cells",
    "NK cells",
    "Neutrophils"
]

plot_df["Display_name"] = pd.Categorical(
    plot_df["Display_name"],
    categories=order,
    ordered=True
)

plot_df = (
    plot_df
    .sort_values("Display_name")
    .reset_index(drop=True)
)

y = np.arange(len(plot_df))

fig, ax = plt.subplots(
    figsize=(8.5, 5.8)
)

offset = 0.13


# ssGSEA
ax.scatter(
    plot_df["ssGSEA_Risk_rho"],
    y + offset,
    s=85,
    label="ssGSEA",
    zorder=3
)

# MCP-counter
ax.scatter(
    plot_df["MCP_Risk_rho"],
    y - offset,
    s=85,
    marker="s",
    label="MCP-counter",
    zorder=3
)


# 同一细胞两种算法连线
for i, row in plot_df.iterrows():

    ax.plot(
        [
            row["ssGSEA_Risk_rho"],
            row["MCP_Risk_rho"]
        ],
        [
            i + offset,
            i - offset
        ],
        linewidth=1.2
    )


# 0线
ax.axvline(
    0,
    linestyle="--",
    linewidth=1
)


# y轴
ax.set_yticks(y)

ax.set_yticklabels(
    plot_df["Display_name"],
    fontsize=10
)


# x轴
ax.set_xlabel(
    "Spearman correlation with RiskScore",
    fontsize=11
)

ax.set_title(
    "Cross-method validation of immune-cell associations",
    fontsize=14,
    pad=12
)


# x轴范围
xmin = min(
    plot_df["ssGSEA_Risk_rho"].min(),
    plot_df["MCP_Risk_rho"].min()
) - 0.05

xmax = max(
    plot_df["ssGSEA_Risk_rho"].max(),
    plot_df["MCP_Risk_rho"].max()
) + 0.05

ax.set_xlim(
    xmin,
    xmax
)


# legend
ax.legend(
    frameon=False,
    loc="best"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "ssGSEA_vs_MCPcounter_RiskScore_Concordance.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "ssGSEA_vs_MCPcounter_RiskScore_Concordance.pdf"
    ),
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.87
# Cross-method validation summary
# ============================================================

summary_rows = []


for _, row in cross_method.iterrows():

    # High vs Low方向
    ss_group_dir = (
        "Higher in High-risk"
        if row["ssGSEA_High_minus_Low"] > 0
        else "Lower in High-risk"
    )

    mcp_group_dir = (
        "Higher in High-risk"
        if row["MCP_High_minus_Low"] > 0
        else "Lower in High-risk"
    )


    # Group validation
    if (
        row["Group_direction_consistent"]
        and row["ssGSEA_Group_FDR"] < 0.05
        and row["MCP_Group_FDR"] < 0.05
    ):
        group_status = "Validated"

    elif row["Group_direction_consistent"]:
        group_status = "Direction consistent"

    else:
        group_status = "Discordant"


    # RiskScore correlation validation
    if (
        row["Risk_direction_consistent"]
        and row["ssGSEA_Risk_FDR"] < 0.05
        and row["MCP_Risk_FDR"] < 0.05
    ):
        corr_status = "Validated"

    elif row["Risk_direction_consistent"]:
        corr_status = "Direction consistent"

    else:
        corr_status = "Discordant"


    summary_rows.append({

        "Immune_feature":
            row["Immune_feature"],

        "ssGSEA_Group_direction":
            ss_group_dir,

        "MCP_Group_direction":
            mcp_group_dir,

        "Group_validation":
            group_status,

        "ssGSEA_Risk_rho":
            row["ssGSEA_Risk_rho"],

        "MCP_Risk_rho":
            row["MCP_Risk_rho"],

        "RiskScore_validation":
            corr_status
    })


validation_summary = pd.DataFrame(
    summary_rows
)

display(validation_summary)

In [ ]:
validation_summary.to_csv(
    os.path.join(
        result_dir,
        "LIHC_Immune_CrossMethod_Validation_Summary.csv"
    ),
    index=False
)

print("Saved.")